#### setup

In [ ]:
# make library code importable
import sys
from pathlib import Path
ROOT = Path.cwd().parents[1]
sys.path.append(str(ROOT / "library"))

# generic imports
import os
import json
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader

# library imports
from data_utils import *
from models import *
from training import *
from eval_utils import *

In [ ]:
# set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# set confounders
confounders = ['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10','f11']
input_dim = len(confounders)

#### helpers

In [ ]:
def load_nuisance_models(ns_seed_dir, input_dim, device):
    """Helper to load nuisance models. Set the hidden dim correctly!"""
    paths = {
        "e": ns_seed_dir / "prop_model.pt",
        "m0": ns_seed_dir / "mu0_model.pt",
        "m1": ns_seed_dir / "mu1_model.pt"}

    for name, path in paths.items():
        if not path.exists():
            raise FileNotFoundError(f"Missing nuisance checkpoint {name}: {path}")

    prop_model = ClassificationHead(input_dim=input_dim, hidden_dim=64).to(device)
    prop_model.load_state_dict(torch.load(paths["e"], map_location=device, weights_only=True))

    m0_model = ClassificationHead(input_dim=input_dim, hidden_dim=128).to(device)
    m0_model.load_state_dict(torch.load(paths["m0"], map_location=device, weights_only=True))

    m1_model = ClassificationHead(input_dim=input_dim, hidden_dim=64).to(device)
    m1_model.load_state_dict(torch.load(paths["m1"], map_location=device, weights_only=True))

    return prop_model, m0_model, m1_model

In [ ]:
@torch.no_grad()
def score_binary_response(df, confounders, model, device):
    batch_size=1024
    model.eval()
    xs = torch.tensor(df[confounders].astype(np.float32).values, dtype=torch.float32)
    preds = []
    for i in range(0, len(xs), batch_size):
        x = xs[i:i+batch_size].to(device)
        logits = model(x)
        preds.append(torch.sigmoid(logits).detach().cpu())
    return torch.cat(preds).numpy()

In [ ]:
def compute_dr_scores(df, confounders, prop_model, m0_model, m1_model, device):

    # score nuisances
    e_hat  = score_propensity(df, confounders, prop_model, device).astype(np.float32).ravel()
    m0_hat = score_binary_response(df, confounders, m0_model, device).astype(np.float32).ravel()
    m1_hat = score_binary_response(df, confounders, m1_model, device).astype(np.float32).ravel()
    
    # DR components
    T = df["T"].astype(np.float32).to_numpy()
    Y = df["Y"].astype(np.float32).to_numpy()

    # stabilize denominator
    eps = 1e-3
    denom_e   = np.clip(e_hat,   eps, 1.0 - eps)
    denom_1_e = np.clip(1.0 - e_hat, eps, 1.0 - eps)

    # compute DR scores
    term_treated = (T * (Y - m1_hat)) / denom_e
    term_control = ((1.0 - T) * (Y - m0_hat)) / denom_1_e
    dr = term_treated - term_control + (m1_hat - m0_hat)

    # store
    df["e_hat"]  = e_hat
    df["m0_hat"] = m0_hat
    df["m1_hat"] = m1_hat
    df['DR'] = dr.astype(np.float32)

    return df

#### train and store

In [ ]:
# load configs
configs = pd.read_csv(f'./configs/criteo_pointwise.csv', index_col=0)

# set configs
row = configs.iloc[0]
params = dict(
    hidden_dim=int(row["hidden_dim"]),
    learning_rate=float(row["lr"]),
    weight_decay=float(row["weight_decay"]),
    batch_size=int(row["batch_size"]),
    max_epochs=50,
    patience=5)

In [ ]:
# set directories
out_dir = f'./chkpts/pointwise/'
os.makedirs(out_dir, exist_ok=True)

ns_dir = ROOT / "experiments" / "criteo" / "chkpts" / "nuisances"

In [ ]:
# loop over seeds
for seed in range(5):

    # track progress
    print(f" -> Seed {seed}")
    set_seed(seed)

    # set output dir
    ckpt_dir = Path(out_dir) / f"seed_{seed}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    # get training data
    train_df = pd.read_csv(ROOT / "experiments" / "criteo" / "data" / "datasets" / f"seed_{seed}" / "train_2.csv")
    val_df = pd.read_csv(ROOT / "experiments" / "criteo" / "data" / "datasets" / f"seed_{seed}" / "val_2.csv")

    # load nuisance models
    ns_seed_dir = ns_dir / f"seed_{seed}"
    prop_model, m0_model, m1_model = load_nuisance_models(ns_seed_dir=ns_seed_dir, input_dim=input_dim, device=device)

    # add pseudo outcomes to dataframes
    train_df = compute_dr_scores(train_df, confounders, prop_model, m0_model, m1_model, device)
    val_df = compute_dr_scores(val_df, confounders, prop_model, m0_model, m1_model, device)

    # clip for stability using training data
    c = train_df["DR"].abs().quantile(0.99)
    train_df["DR"] = train_df["DR"].clip(lower=-c, upper=c)
    val_df["DR"] = val_df["DR"].clip(lower=-c, upper=c)

    # make data loaders
    train_loader, val_loader = make_cate_loaders(train_df, val_df, confounders, batch_size=params["batch_size"])

    # init model
    cate_model = RegressionHead(input_dim=input_dim, hidden_dim=params["hidden_dim"]).to(device)

    # train model
    cate_model, info_phi = train_cate(cate_model, train_loader, val_loader, device, lr=params["learning_rate"], weight_decay=params["weight_decay"],
                                      max_epochs=50, patience=5, seed=seed)

    # checkpoint
    torch.save(cate_model.state_dict(), ckpt_dir / "cate_model.pt")